In [ ]:
# CELL 1: Define project paths, conda environment name, and helper functions
# This notebook is intended to be run from VS Code on Linux or WSL2/Ubuntu.
# It does NOT require the notebook kernel itself to be the VIBE environment,
# because all VIBE commands are executed through `conda run -n vibe-env ...`.

from pathlib import Path
import os
import shlex
import subprocess
import sys

# ---- USER CONFIGURATION ----
PROJECT_ROOT = Path.home() / "smpl_vibe_project"
REPO_URL = "https://github.com/mkocabas/VIBE.git"
REPO_DIR = PROJECT_ROOT / "VIBE"
ENV_NAME = "vibe-env"

# For first validation, use VIBE's sample video downloaded by scripts/prepare_data.sh.
# Later, replace this with your own .mp4 path.
INPUT_VIDEO = REPO_DIR / "sample_video.mp4"
OUTPUT_DIR = REPO_DIR / "output"

# Keep this False for the first test. Rendering needs EGL/OpenGL support.
RENDER_OUTPUT = False

# Safe default for old/limited GPUs. Increase later if memory allows.
TRACKER_BATCH_SIZE = 2
VIBE_BATCH_SIZE = 64

def run(cmd, cwd=None, check=True, env=None):
    """Run a shell command and stream output."""
    if isinstance(cmd, (list, tuple)):
        printable = " ".join(shlex.quote(str(x)) for x in cmd)
    else:
        printable = cmd
    print("\n$ " + printable)
    merged_env = os.environ.copy()
    if env:
        merged_env.update({str(k): str(v) for k, v in env.items()})
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        shell=isinstance(cmd, str),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=merged_env,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}:\n{printable}")
    return result

def conda_run(command, cwd=None, check=True, extra_env=None):
    """Run a command inside the VIBE conda environment."""
    env_exports = ""
    if extra_env:
        env_exports = " ".join(f"{k}={shlex.quote(str(v))}" for k, v in extra_env.items()) + " "
    cmd = f"conda run -n {shlex.quote(ENV_NAME)} bash -lc {shlex.quote(env_exports + command)}"
    return run(cmd, cwd=cwd, check=check)

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPO_DIR:", REPO_DIR)
print("ENV_NAME:", ENV_NAME)

In [ ]:
# CELL 2: Check basic system dependencies before installing VIBE
# Required: conda, git, ffmpeg. Recommended for GPU: nvidia-smi.

checks = [
    "uname -a || ver",
    "which conda && conda --version",
    "which git && git --version",
    "which ffmpeg && ffmpeg -version | head -n 1",
    "nvidia-smi || true",
]

for c in checks:
    run(c, check=False)

In [ ]:
# CELL 3: Clone the official VIBE repository and record the commit
# If the repository already exists, this updates it with git pull.

if not REPO_DIR.exists():
    run(f"git clone {shlex.quote(REPO_URL)} {shlex.quote(str(REPO_DIR))}", cwd=PROJECT_ROOT)
else:
    run("git fetch --all --prune", cwd=REPO_DIR, check=False)
    run("git pull", cwd=REPO_DIR, check=False)

run("git rev-parse HEAD", cwd=REPO_DIR)
run("ls -la", cwd=REPO_DIR)

In [ ]:
# CELL 4: Create the exact legacy conda environment for VIBE
# Official VIBE uses Python 3.7, PyTorch 1.4.0, torchvision 0.5.0, and old pinned packages.
# Do not replace these with latest PyTorch unless you are intentionally porting the code.

envs = run("conda env list", check=False).stdout
if ENV_NAME not in envs:
    run(f"conda create -y -n {shlex.quote(ENV_NAME)} python=3.7 pip")
else:
    print(f"Conda environment {ENV_NAME!r} already exists.")

# Old packages install more reliably with a not-too-new pip/setuptools stack.
conda_run("python -m pip install --upgrade 'pip<24' 'setuptools<60' wheel", cwd=REPO_DIR)

# Official VIBE installation core.
# This normally installs a CUDA-enabled PyTorch 1.4 wheel on Linux.
conda_run("python -m pip install numpy==1.17.5 torch==1.4.0 torchvision==0.5.0", cwd=REPO_DIR)

# Pytube version from the official script.
conda_run("python -m pip install 'git+https://github.com/giacaglia/pytube.git' --upgrade", cwd=REPO_DIR)

# Remaining official requirements.
conda_run("python -m pip install -r requirements.txt", cwd=REPO_DIR)

In [ ]:
# CELL 5: Install optional Linux runtime libraries needed by rendering and video processing
# Run this cell only if you have sudo access. If not, skip it and first run VIBE with --no_render.

print("""
Optional system packages for Ubuntu/WSL2/headless rendering:

sudo apt-get update
sudo apt-get install -y ffmpeg libgl1 libglib2.0-0 libegl1-mesa libgles2-mesa mesa-utils

If you do not have sudo rights, ask the system administrator to install them.
For the first smoke test this notebook uses --no_render, so EGL/OpenGL is less critical.
""")

In [ ]:
# CELL 6: Download VIBE pretrained data, SMPL data used by the repo, sample video, and YOLO weights
# This calls the official scripts/prepare_data.sh.
# It downloads data/vibe_data.zip via gdown, unzips it, moves sample_video.mp4 to the repo root,
# and places yolov3.weights under ~/.torch/models/.

conda_run("bash scripts/prepare_data.sh", cwd=REPO_DIR)

# Show the most important files after preparation.
run("find data/vibe_data -maxdepth 2 -type f | sort | sed -n '1,80p'", cwd=REPO_DIR)
run("ls -lh sample_video.mp4 || true", cwd=REPO_DIR)
run("ls -lh $HOME/.torch/models/yolov3.weights || true", cwd=REPO_DIR)

In [ ]:
# CELL 7: Verify Python imports and required VIBE assets
# This cell detects missing files before running the demo.

verify_script = r"""
from pathlib import Path
import torch
import torchvision
import cv2
import numpy as np
import smplx
import joblib

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))

root = Path(".")
required = [
    "sample_video.mp4",
    "data/vibe_data/vibe_model_wo_3dpw.pth.tar",
    "data/vibe_data/spin_model_checkpoint.pth.tar",
    "data/vibe_data/J_regressor_extra.npy",
    "data/vibe_data/smpl_mean_params.npz",
]
for rel in required:
    p = root / rel
    print(f"{rel}: {'OK' if p.exists() else 'MISSING'}")

smpl_files = list((root / "data" / "vibe_data").glob("*.pkl")) + list((root / "data" / "vibe_data").glob("SMPL*"))
print("SMPL-like files:", [str(p) for p in smpl_files])
"""
conda_run(f"python - <<'PY'\n{verify_script}\nPY", cwd=REPO_DIR)

In [ ]:
# CELL 8: Run the official VIBE demo in non-rendering mode
# This is the best first smoke test because it avoids most OpenGL/EGL issues.
# It should produce output/sample_video/vibe_output.pkl.

demo_cmd = (
    f"python demo.py "
    f"--vid_file {shlex.quote(str(INPUT_VIDEO.name))} "
    f"--output_folder {shlex.quote(str(OUTPUT_DIR.name))} "
    f"--tracking_method bbox "
    f"--detector yolo "
    f"--tracker_batch_size {TRACKER_BATCH_SIZE} "
    f"--vibe_batch_size {VIBE_BATCH_SIZE} "
    f"--no_render"
)

conda_run(
    demo_cmd,
    cwd=REPO_DIR,
    extra_env={
        "PYTHONPATH": str(REPO_DIR),
        "PYOPENGL_PLATFORM": "egl",
    }
)

run("find output -maxdepth 3 -type f | sort", cwd=REPO_DIR)

In [ ]:
# CELL 9: Optionally run VIBE with rendered video output
# Enable RENDER_OUTPUT=True in CELL 1 only after CELL 8 works.
# Rendering can be slow and may require EGL/OpenGL libraries.

if RENDER_OUTPUT:
    render_cmd = (
        f"python demo.py "
        f"--vid_file {shlex.quote(str(INPUT_VIDEO.name))} "
        f"--output_folder {shlex.quote(str(OUTPUT_DIR.name))} "
        f"--tracking_method bbox "
        f"--detector yolo "
        f"--tracker_batch_size {TRACKER_BATCH_SIZE} "
        f"--vibe_batch_size {VIBE_BATCH_SIZE} "
        f"--sideview "
        f"--smooth"
    )
    conda_run(
        render_cmd,
        cwd=REPO_DIR,
        extra_env={
            "PYTHONPATH": str(REPO_DIR),
            "PYOPENGL_PLATFORM": "egl",
        }
    )
    run("find output -maxdepth 3 -type f | sort", cwd=REPO_DIR)
else:
    print("RENDER_OUTPUT is False. Keeping the deployment test in --no_render mode.")

In [ ]:
# CELL 10: Inspect the VIBE output pickle
# Expected output keys per person include:
# pred_cam, orig_cam, verts, pose, betas, joints3d, joints2d, joints2d_img_coord, bboxes, frame_ids.

inspect_script = r"""
from pathlib import Path
import joblib

pkl_files = sorted(Path("output").glob("*/vibe_output.pkl"))
if not pkl_files:
    raise FileNotFoundError("No vibe_output.pkl found under output/*/")

pkl_path = pkl_files[-1]
print("Inspecting:", pkl_path)

data = joblib.load(pkl_path)
print("Track/person IDs:", list(data.keys()))

for person_id, person_data in data.items():
    print("\\nPERSON", person_id)
    for key, value in person_data.items():
        shape = getattr(value, "shape", None)
        print(f"  {key:20s}: {shape if shape is not None else type(value)}")
"""
conda_run(f"python - <<'PY'\n{inspect_script}\nPY", cwd=REPO_DIR)

In [ ]:
# CELL 11: Export a compact CSV summary for documentation/comparison
# This creates output/<video>/vibe_summary.csv with frame IDs, bbox values, camera values, and beta values.

export_script = r"""
from pathlib import Path
import joblib
import csv

pkl_files = sorted(Path("output").glob("*/vibe_output.pkl"))
if not pkl_files:
    raise FileNotFoundError("No vibe_output.pkl found under output/*/")

pkl_path = pkl_files[-1]
out_csv = pkl_path.parent / "vibe_summary.csv"

data = joblib.load(pkl_path)

header = [
    "person_id", "local_index", "frame_id",
    "bbox_cx", "bbox_cy", "bbox_w", "bbox_h",
    "orig_cam_sx", "orig_cam_sy", "orig_cam_tx", "orig_cam_ty",
] + [f"beta_{i}" for i in range(10)]

rows = []
for person_id, person_data in data.items():
    frames = person_data["frame_ids"]
    bboxes = person_data["bboxes"]
    orig_cam = person_data["orig_cam"]
    betas = person_data["betas"]
    for i in range(len(frames)):
        rows.append(
            [person_id, i, int(frames[i])]
            + [float(x) for x in bboxes[i].tolist()]
            + [float(x) for x in orig_cam[i].tolist()]
            + [float(x) for x in betas[i].tolist()]
        )

with open(out_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(rows)

print("Wrote:", out_csv)
print("Rows:", len(rows))
"""
conda_run(f"python - <<'PY'\n{export_script}\nPY", cwd=REPO_DIR)
run("find output -name 'vibe_summary.csv' -print", cwd=REPO_DIR)

In [ ]:
# CELL 12: Optional official 3DPW evaluation preparation
# Use this only after you have legally downloaded 3DPW and placed it under:
#   VIBE/data/3dpw/imageFiles
#   VIBE/data/3dpw/sequenceFiles/test
#   VIBE/data/3dpw/sequenceFiles/train
#   VIBE/data/3dpw/sequenceFiles/validation
#
# Then this cell preprocesses 3DPW and runs eval.py with the official pretrained checkpoint.

three_dpw_root = REPO_DIR / "data" / "3dpw"
three_dpw_ready = (three_dpw_root / "imageFiles").exists() and (three_dpw_root / "sequenceFiles" / "test").exists()

if three_dpw_ready:
    print("3DPW detected. Preparing VIBE DB files...")
    conda_run("export PYTHONPATH=./:$PYTHONPATH && python lib/data_utils/threedpw_utils.py --dir ./data/3dpw", cwd=REPO_DIR)

    make_cfg = r"""
from pathlib import Path
import yaml

base = Path("configs/config.yaml")
out = Path("configs/config_eval_vibe.yaml")

cfg = yaml.safe_load(base.read_text())
cfg["TRAIN"]["PRETRAINED"] = "data/vibe_data/vibe_model_wo_3dpw.pth.tar"
cfg["TRAIN"]["BATCH_SIZE"] = 16
cfg["NUM_WORKERS"] = 4

out.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Wrote", out)
"""
    conda_run(f"python - <<'PY'\n{make_cfg}\nPY", cwd=REPO_DIR)
    conda_run("export PYTHONPATH=./:$PYTHONPATH && python eval.py --cfg configs/config_eval_vibe.yaml", cwd=REPO_DIR)
else:
    print("3DPW is not detected in the expected folder. Skipping official evaluation.")
    print("Expected:", three_dpw_root)

In [ ]:
# CELL 13: Create reusable metric utilities for your P9 comparison
# These functions can be used later when you have predicted arrays and ground-truth arrays aligned.

metrics_code = """
import numpy as np

def mpjpe(pred_joints, gt_joints):
    \"\"\"Mean Per-Joint Position Error. pred/gt shape: (..., J, 3).\"\"\"
    pred = np.asarray(pred_joints)
    gt = np.asarray(gt_joints)
    return float(np.mean(np.linalg.norm(pred - gt, axis=-1)))

def pve(pred_vertices, gt_vertices):
    \"\"\"Per-Vertex Error. pred/gt shape: (..., V, 3).\"\"\"
    pred = np.asarray(pred_vertices)
    gt = np.asarray(gt_vertices)
    return float(np.mean(np.linalg.norm(pred - gt, axis=-1)))

def beta_l2(pred_betas, gt_betas):
    \"\"\"L2 distance between SMPL beta vectors. pred/gt shape: (..., 10).\"\"\"
    pred = np.asarray(pred_betas)
    gt = np.asarray(gt_betas)
    return float(np.mean(np.linalg.norm(pred - gt, axis=-1)))

def pck(pred_joints, gt_joints, threshold=0.15):
    \"\"\"Percentage of Correct Keypoints under a distance threshold in meters.\"\"\"
    pred = np.asarray(pred_joints)
    gt = np.asarray(gt_joints)
    dist = np.linalg.norm(pred - gt, axis=-1)
    return float(np.mean(dist < threshold))

def compute_similarity_transform(S1, S2):
    \"\"\"
    Procrustes alignment from S1 to S2.
    S1/S2 shape: (N, 3), where N is number of joints or vertices.
    \"\"\"
    S1 = np.asarray(S1).T
    S2 = np.asarray(S2).T

    mu1 = S1.mean(axis=1, keepdims=True)
    mu2 = S2.mean(axis=1, keepdims=True)
    X1 = S1 - mu1
    X2 = S2 - mu2

    var1 = np.sum(X1 ** 2)
    K = X1 @ X2.T

    U, s, Vh = np.linalg.svd(K)
    V = Vh.T
    Z = np.eye(U.shape[0])
    Z[-1, -1] = np.sign(np.linalg.det(U @ V.T))

    R = V @ Z @ U.T
    scale = np.trace(R @ K) / var1
    t = mu2 - scale * R @ mu1

    S1_hat = scale * R @ S1 + t
    return S1_hat.T

def pa_mpjpe(pred_joints, gt_joints):
    \"\"\"Procrustes Aligned MPJPE. pred/gt shape: (F, J, 3) or (J, 3).\"\"\"
    pred = np.asarray(pred_joints)
    gt = np.asarray(gt_joints)

    if pred.ndim == 2:
        pred_aligned = compute_similarity_transform(pred, gt)
        return float(np.mean(np.linalg.norm(pred_aligned - gt, axis=-1)))

    errors = []
    for p, g in zip(pred, gt):
        p_aligned = compute_similarity_transform(p, g)
        errors.append(np.mean(np.linalg.norm(p_aligned - g, axis=-1)))
    return float(np.mean(errors))

def acceleration_error(pred_joints, gt_joints):
    \"\"\"
    Acceleration error for temporal consistency.
    pred/gt shape: (F, J, 3). Requires at least 3 frames.
    \"\"\"
    pred = np.asarray(pred_joints)
    gt = np.asarray(gt_joints)

    if pred.shape[0] < 3:
        return float("nan")

    pred_acc = pred[:-2] - 2 * pred[1:-1] + pred[2:]
    gt_acc = gt[:-2] - 2 * gt[1:-1] + gt[2:]
    return float(np.mean(np.linalg.norm(pred_acc - gt_acc, axis=-1)))
"""

tools_dir = REPO_DIR / "tools"
tools_dir.mkdir(exist_ok=True)
metrics_path = tools_dir / "p9_metrics.py"
metrics_path.write_text(metrics_code)
print("Wrote:", metrics_path)

In [ ]:
# CELL 14: Write a deployment report JSON for your final practice documentation
# This records repository commit, environment versions, GPU availability, and output files.

report_script = r"""
from pathlib import Path
import subprocess
import json
import torch
import torchvision
import sys
import glob

def cmd_out(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.STDOUT).strip()
    except Exception as e:
        return str(e)

report = {
    "method": "VIBE",
    "repo_commit": cmd_out("git rev-parse HEAD"),
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "vibe_outputs": sorted(glob.glob("output/**/*", recursive=True)),
    "notes": [
        "First successful deployment criterion: demo.py runs and creates output/<video>/vibe_output.pkl",
        "For quantitative evaluation, download 3DPW separately and run CELL 12.",
        "For fair P9 comparison, record FPS, GPU model, checkpoint, dataset split, and exact commit."
    ],
}

out = Path("deployment_report_vibe.json")
out.write_text(json.dumps(report, indent=2))
print(out.read_text())
"""
conda_run(f"python - <<'PY'\n{report_script}\nPY", cwd=REPO_DIR)

In [ ]:
# CELL 15: Common troubleshooting commands
# Run these when something fails; paste the output into your deployment report.

troubleshooting_commands = [
    "conda run -n vibe-env python --version",
    "conda run -n vibe-env python -c 'import torch; print(torch.__version__, torch.cuda.is_available())'",
    "conda run -n vibe-env python -c 'import cv2, smplx, pyrender, trimesh; print(\"imports ok\")'",
    "find data/vibe_data -maxdepth 1 -type f -printf '%f\n' | sort",
    "find output -maxdepth 3 -type f | sort",
]

for c in troubleshooting_commands:
    run(c, cwd=REPO_DIR, check=False)